# HadEX3 climate extremes: India

The [HadEX3](https://www.metoffice.gov.uk/hadobs/hadex3/) dataset from the Met
Office Hadley Centre provides global gridded climate extremes based on 29 ETCCDI
indices at 1.25 x 1.875 degree resolution over 1901-2018
(Dunn et al. 2020, doi:10.1029/2019JD032263).

This notebook shows how to query and visualize annual TXx (hottest day) and
Rx1day (maximum 1-day rainfall) trends over India, seasonal cycles from monthly
data, and decadal spatial patterns.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import varunayan as v
from varunayan.hadex3 import list_hadex3_indices, hadex3_bbox

## Available indices

In [ ]:
indices = list_hadex3_indices()
indices.head(12)

## Download data for India

We fetch annual TXx and Rx1day for 1951-2018, plus monthly TXx for 1981-2018.
Each call downloads a ~15 MB gzipped NetCDF from the Met Office and caches it
locally.

In [ ]:
india_bbox = dict(north=37, south=6, east=98, west=68)

india_txx = hadex3_bbox(
    index="TXx", start_year=1951, end_year=2018,
    frequency="annual", **india_bbox,
)

india_rx1day = hadex3_bbox(
    index="Rx1day", start_year=1951, end_year=2018,
    frequency="annual", **india_bbox,
)

india_txx_mon = hadex3_bbox(
    index="TXx", start_year=1981, end_year=2018,
    frequency="monthly", **india_bbox,
)

print(f"TXx annual: {len(india_txx)} rows")
print(f"Rx1day annual: {len(india_rx1day)} rows")
print(f"TXx monthly: {len(india_txx_mon)} rows")

## Annual temperature extremes trend

Spatial mean of annual TXx (the hottest day) across India's grid cells with a
LOWESS trend.

In [ ]:
from statsmodels.nonparametric.smoothers_lowess import lowess

txx_annual = india_txx.groupby("year")["value"].mean().reset_index()

smooth = lowess(txx_annual["value"], txx_annual["year"], frac=0.3)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(txx_annual["year"], txx_annual["value"], color="#BDBDBD", lw=0.8)
ax.plot(smooth[:, 0], smooth[:, 1], color="#B2182B", lw=1.5)
ax.set_xlabel(None)
ax.set_ylabel("TXx (\u00b0C)")
ax.set_title("India: Annual Maximum Temperature (TXx), 1951-2018\n"
             "Spatial mean across grid cells; line = LOWESS trend")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Seasonal patterns from monthly data

Averaging monthly TXx over 1981-2018 shows the pre-monsoon peak in May-June and
the cooler post-monsoon months.

In [ ]:
import calendar

seasonal = india_txx_mon.groupby("month")["value"].mean().reset_index()
month_labels = [calendar.month_abbr[m] for m in seasonal["month"]]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(seasonal["month"], seasonal["value"], color="#B2182B", lw=1.5, marker="o", ms=5)
ax.set_xticks(seasonal["month"])
ax.set_xticklabels(month_labels)
ax.set_ylabel("Mean TXx (\u00b0C)")
ax.set_title("India: Seasonal Cycle of TXx (1981-2018 mean)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Extreme rainfall trend (Rx1day)

Annual maximum 1-day precipitation over India 1951-2018.

In [ ]:
rx1_annual = india_rx1day.groupby("year")["value"].mean().reset_index()

smooth_rx = lowess(rx1_annual["value"], rx1_annual["year"], frac=0.35)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(rx1_annual["year"], rx1_annual["value"], color="#BDBDBD", lw=0.8)
ax.plot(smooth_rx[:, 0], smooth_rx[:, 1], color="#2166AC", lw=1.5)
ax.set_ylabel("Rx1day (mm)")
ax.set_title("India: Annual Maximum 1-Day Rainfall (Rx1day), 1951-2018\n"
             "Spatial mean; line = LOWESS trend")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Decadal spatial distribution of TXx

Comparing the spatial pattern of TXx across three periods.

In [ ]:
bins = [1950, 1970, 1990, 2018]
labels = ["1951-1970", "1971-1990", "1991-2018"]
india_txx["decade"] = pd.cut(india_txx["year"], bins=bins, labels=labels, include_lowest=True)

decades = india_txx.groupby(["decade", "latitude", "longitude"])["value"].mean().reset_index()

fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharex=True, sharey=True)
vmin = decades["value"].quantile(0.02)
vmax = decades["value"].quantile(0.98)

for ax, label in zip(axes, labels):
    sub = decades[decades["decade"] == label]
    sc = ax.scatter(
        sub["longitude"], sub["latitude"], c=sub["value"],
        cmap="RdYlBu_r", vmin=vmin, vmax=vmax, s=20, edgecolors="none",
    )
    ax.set_title(label)
    ax.set_aspect("equal")

axes[0].set_ylabel("Latitude")
for ax in axes:
    ax.set_xlabel("Longitude")

fig.colorbar(sc, ax=axes, label="\u00b0C", shrink=0.8)
fig.suptitle("India: Mean Annual TXx by Decade", fontsize=13)
plt.tight_layout()
plt.show()